In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun #requires this for arxiv query run tool
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper
from langchain_core.tools import create_retriever_tool

C:\Users\ybalasaraswa\OneDrive - OpenText\Desktop\deep learning\LangChain\.myvenv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\ybalasaraswa\OneDrive - OpenText\Desktop\deep learning\LangChain\.myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=10, doc_content_chars_max=500)
arxiv_api_wrapper = ArxivAPIWrapper(top_k_results=10, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=wiki_api_wrapper)
arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)
print("tools currently available: ",wiki.name, "& ", arxiv.name)

tools currently available:  wikipedia &  arxiv


In [3]:
tools = [wiki, arxiv]

In [4]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
loader = WebBaseLoader("https://docs.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectorDB = FAISS.from_documents(documents, OllamaEmbeddings(model="llama3.1"))
retrival = vectorDB.as_retriever()
# print(f"Number of documents loaded: {len(documents)}")

C:\Users\ybalasaraswa\AppData\Local\Temp\ipykernel_44316\3774912514.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  vectorDB = FAISS.from_documents(documents, OllamaEmbeddings(model="llama3.1"))


In [6]:
retrival_tool = create_retriever_tool(
    retriever=retrival,
    name="LangChainDocsRetriever",
    description="useful for retrieving information from the LangChain documentation"
)

In [7]:
tools.append(retrival_tool)

In [8]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'C:\\Users\\ybalasaraswa\\OneDrive - OpenText\\Desktop\\deep learning\\LangChain\\.myvenv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=10, lang='en', load_all_available_meta=False, doc_content_chars_max=500)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=10, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=500)),
 StructuredTool(name='LangChainDocsRetriever', description='useful for retrieving information from the LangChain documentation', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001D67B52B1C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000

## now Run all the tools with agents and LLM models

## Tools, LLM --> AgentExecutor --> AgentExecutorWithTools

## we will be using Groq for this too

In [9]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

model = ChatGroq(api_key=groq_api_key,
                 model="openai/gpt-oss-120b")

In [10]:
## prompt Template
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client
client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent")
print(prompt.messages)


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [11]:
#Agents
from langchain_classic.agents import create_openai_tools_agent

agent = create_openai_tools_agent(llm=model,tools=tools,prompt=prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat

In [12]:
#agent executor
from langchain_classic.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='

In [13]:
agent_executor.invoke({"input":"tell me about langsmith"})



> Entering new AgentExecutor chain...

Invoking: `LangChainDocsRetriever` with `{'query': 'LangSmith'}`


Edit this page on GitHub or file an issue.
Connect these docs to Claude, VSCode, and more via MCP for real-time answers.⌘I

moreDeep Agents (TypeScript)Quickly build agents that can tackle any task.Learn moreLangChain (TypeScript)Customize the building blocks of your agents.Learn moreLangGraph (TypeScript)Control every step of your custom agent with low-level orchestration, memory, and human-in-the-loop support.Learn moreGet startedBuild your first agent with LangChainGet startedSign up for LangSmithTry LangSmithBuild an advanced agent with LangGraphGet startedEnroll in LangChain AcademyGet startedAdditional resourcesCommunity forumAsk questions, share solutions, and discuss best practices.JoinCommunity SlackConnect with other builders and get quick help.JoinSupport portalSubmit tickets and track support requests.VisitLangSmith statusReal-time status of LangSmith services and API

{'input': 'tell me about langsmith',
 'output': '**LangSmith\u202f—\u202fThe “DevOps” platform for AI agents**\n\nLangSmith is a SaaS product built by the LangChain team that gives AI teams the tooling they need to **observe, test, improve, and ship** production‑grade agents and LLM‑powered applications. Think of it as the combination of APM (application‑performance‑monitoring) and CI/CD, but for generative‑AI workloads.\n\n---\n\n## Core Capabilities\n\n| Area | What LangSmith Provides | Why It Matters |\n|------|------------------------|----------------|\n| **Observability** | • End‑to‑end tracing of every LLM call, tool use, and custom code execution  <br>• Real‑time visualizations of the “thought process” (prompt → LLM → tool → output)  <br>• Aggregate metrics (latency, token usage, error rates) and trend dashboards | Gives you visibility into why an agent behaved a certain way, helps you spot regressions, and lets you quantify cost and performance. |\n| **Evaluation** | • Built‑in

In [14]:
agent_executor.invoke({"input":"what is deep learning?"})



> Entering new AgentExecutor chain...
**Deep learning** is a sub‑field of machine learning that models high‑level abstractions in data by using **artificial neural networks with many layers** (hence “deep”).  

---

## Core Idea
- **Neural networks** are composed of interconnected “neurons” that apply linear transformations followed by non‑linear activation functions.
- **Depth** means stacking multiple hidden layers, allowing the network to learn hierarchical representations: lower layers capture simple patterns (edges, tones), while higher layers combine them into more abstract concepts (objects, semantics, strategies).

## How It Works
1. **Input** – raw data (pixels, audio waveforms, text tokens, sensor readings, etc.).
2. **Forward pass** – data propagates through each layer, being transformed by weights and activations.
3. **Loss calculation** – a cost function measures how far the network’s output is from the desired target.
4. **Backpropagation** – gradients of the loss with 

{'input': 'what is deep learning?',
 'output': '**Deep learning** is a sub‑field of machine learning that models high‑level abstractions in data by using **artificial neural networks with many layers** (hence “deep”).  \n\n---\n\n## Core Idea\n- **Neural networks** are composed of interconnected “neurons” that apply linear transformations followed by non‑linear activation functions.\n- **Depth** means stacking multiple hidden layers, allowing the network to learn hierarchical representations: lower layers capture simple patterns (edges, tones), while higher layers combine them into more abstract concepts (objects, semantics, strategies).\n\n## How It Works\n1. **Input** – raw data (pixels, audio waveforms, text tokens, sensor readings, etc.).\n2. **Forward pass** – data propagates through each layer, being transformed by weights and activations.\n3. **Loss calculation** – a cost function measures how far the network’s output is from the desired target.\n4. **Backpropagation** – gradien

In [15]:
agent_executor.invoke({"input":"hay what is the research paper about arXiv:1706.03762"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, base**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
**Published:** June 2017 (original conference: NIPS 2017)

---

### Core Idea
The paper introduces the **Transformer** architecture, a novel neural network design for sequence‑to‑sequence tasks (e.g., machine tra

{'input': 'hay what is the research paper about arXiv:1706.03762',
 'output': '**Paper:** *Attention Is All You Need* (arXiv:1706.03762)  \n**Authors:** Ashish\u202fVaswani, Noam\u202fShazeer, Niki\u202fParmar, Jakob\u202fUszkoreit, Llion\u202fJones, Aidan\u202fN.\u202fGomez, Łukasz\u202fKaiser, Illia\u202fPolosukhin  \n**Published:** June\u202f2017 (original conference: NIPS 2017)\n\n---\n\n### Core Idea\nThe paper introduces the **Transformer** architecture, a novel neural network design for sequence‑to‑sequence tasks (e.g., machine translation) that **dispenses with recurrence and convolution entirely**. Instead, it relies solely on **self‑attention mechanisms** to model dependencies between all positions in a sequence, enabling:\n\n* **Parallelizable computation** – each layer processes all tokens simultaneously, unlike recurrent models that must proceed step‑by‑step.  \n* **Long‑range dependency modeling** – attention can directly link any two positions, regardless of distance, wi